# Proving the EWM Promise — Real V-JEPA × HLLSet Cortex

Mirrors `prove_ewm_grounding.ipynb` from ds-hllset-cortex, with the
DeepSeek-OCR encoder replaced by the V-JEPA encoder.  Two proofs:

1. **Fidelity** — encoder → lattice → predictor round-trips real V-JEPA
   encodings with measurable retention.
2. **One-sided grounding** — an out-of-codebook tid is filtered from output
   but still accumulates TF in the LUT (hallucination detection is
   one-sided: the system can flag what it has never measured, and keeps
   measuring what the gate currently forbids).

In [ ]:
import sys
import pathlib

ROOT = pathlib.Path.cwd()
if not (ROOT / "ewm_jepa").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
from ewm_jepa import JEPAPipeline, Quantizer
import hllset_py
from hllset_cortex import HLLSetFilter
from hllset_cortex.domain import encoding_tokenizer, hllset_from_ids, tid

print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## Proof 1 — fidelity: encoder → lattice → decoder round-trips real encodings

In [ ]:
pipe = JEPAPipeline(codebook_size=2048, seed=0)
video, masks_enc, masks_pred = pipe.make_masks(batch_size=1)
result = pipe.roundtrip(video, masks_enc, masks_pred)

for i, s in enumerate(result.stats):
    print(f"pair {i}:")
    print(f"  input   : {s.input_ids} tids ({s.unique_input_ids} unique)")
    print(f"  gated   : {s.gated_ids} tids, set retention {s.set_retention:.3f}")
    print(f"  ordered : {s.restored_ids} tids, retention {s.retention:.3f}")
    print(f"  fidelity: quant cos {s.quant_cosine:.3f}, pred cos {s.pred_cosine:.3f}")


## Proof 2 — grounding is one-sided

Inject a tid the codebook has never seen (`tid99999`).  The gate filters it
from the materialized output; the LUT nevertheless records it.  Later, if the
codebook expands, that tid materializes at its earned TF — the latent
vocabulary dynamic of `main.tex` appendix.

In [ ]:
from hllset_cortex.grounding import has_hallucination

f = HLLSetFilter()
f.tokenizer = encoding_tokenizer()
f.gate_hllset = pipe.gate            # the codebook world view

stream = pipe.quantize_stream(pipe.encode(video)[0]) + " tid99999"
res = f.process_text(stream)

print("gate filtered tid99999:", "tid99999" not in res.token_strings)
print("LUT recorded tid99999 :",
      any(t == "tid99999" for t, _ in f.lut.ranked_tokens()))
print("latent TF of tid99999 :",
      [tf for t, tf in f.lut.ranked_tokens() if t == "tid99999"])

# Hallucination check: bits with no known token in the LUT
print("has_hallucination     :", has_hallucination(res.hllset, f.lut))


## Finding

The single-HLLSet gate filters out-of-codebook tids from **output** while the
LUT keeps measuring them — grounding is one-sided by construction.  The same
finding held for DeepSeek-OCR BPE IDs; it now holds for V-JEPA quantized
patch encodings.  The EWM boundary is encoding-agnostic.